**Volume 1 workbooks v0.1.0, published 2026-08-19.**


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2, norm
from IPython.display import display

ADA_PLOT_COLORS = {
    'blue': '#3c4b99',
    'red': '#c93f55',
    'lightblue': '#8fa8e8',
    'lightpink': '#e36877',
}

def head(obj, n=6):
    if hasattr(obj, 'head'):
        return obj.head(n)
    return obj[:n]

sns.set_theme(style='whitegrid')


In [ ]:
import numpy as np
from math import sqrt
from scipy.stats import binom, chi2, chisquare, f, hypergeom, nct, norm, poisson, t

if 'ada_set_context' not in globals():
    def ada_set_context(exercise_ref: str) -> None:
        return None


## Exercise 6.1. Before we start


To run the exercises in this workshop, we need to load a few packages.


In [ ]:
# Python equivalent setup for plotting and data manipulation
sns.set_theme(style='whitegrid')


## Exercise 6.2.  Customer satisfaction survey


The client satisfaction results from the manuscript client-satisfaction table


In [ ]:
# Hypothesized population proportions from the base year
p = np.array([0.40, 0.16, 0.44])


Results from the follow-up study


In [ ]:
# Observed frequencies from the follow-up study
observed = np.array([15, 12, 13])
# Sample size
n = sum(observed)


The expected frequencies are therefore obtained by multiplying these
proportions by the sample size.


In [ ]:
# Expected frequencies
expected = n * p


The chi-square test statistic is


In [ ]:
# Chi-square test statistic (manual calculation)
chi_sq = sum((observed - expected)**2 / expected)


Calculation of the critical value


In [ ]:
# Degrees of freedom
df = len(observed) - 1
# Critical value for alpha = 0.05
alpha = 0.05
critical_value = chi2.ppf(1 - alpha, df=df)


Calculation of the p-value


In [ ]:
p_value = chi2.sf(chi_sq, df=df)


Using the built-in test


In [ ]:
(lambda _obs, _p: chisquare(f_obs=_obs, f_exp=_obs.sum() * (_p / _p.sum())))(np.asarray(observed, dtype=float), np.asarray(p, dtype=float))


In [ ]:
digits = np.arange(1, 10)
probabilities = np.log10((digits + 1) / digits)
benford = pd.DataFrame({'Digit': digits, 'Probability': probabilities})
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(benford['Digit'], benford['Probability'], color=ADA_PLOT_COLORS['blue'], width=0.7)
ax.set_xlabel('First digit')
ax.set_ylabel('Probability')
ax.set_ylim(0, 0.35)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
plt.show()


In [ ]:
df = pd.DataFrame({
    'Digit': [1,2,3,4,5,6,7,8,9],
    'Rivers': [31.0,16.4,10.7,11.3,7.2,8.6,5.5,4.2,5.1],
    'AmLeague': [32.7,17.6,12.6,9.8,7.4,6.4,4.9,5.6,3.0],
    'CostData': [32.4,18.8,10.1,10.1,9.8,5.5,4.7,5.5,3.1],
    'ReadersDigest': [33.4,18.5,12.4,7.5,7.1,6.5,5.5,4.9,4.2],
    'MolWgt': [26.7,25.2,15.4,10.8,6.7,5.1,4.1,2.8,3.2],
    'Average': [31.2,19.3,12.2,9.9,7.6,6.4,4.9,4.6,3.7],
})
df_long = df.melt(id_vars=['Digit'], var_name='Series', value_name='Percentage')
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=df_long, x='Digit', y='Percentage', hue='Series', marker='o', ax=ax)
ax.set_xlabel('First digit')
ax.set_ylabel('Percentage')
plt.show()


In [ ]:
if 'probabilities' not in globals():
    digits = np.arange(1, 10)
    probabilities = np.log10((digits + 1) / digits)
else:
    digits = np.arange(1, 10)
benford = 100 * probabilities
average = np.array([31.2, 19.3, 12.2, 9.9, 7.6, 6.4, 4.9, 4.6, 3.7])
df = pd.DataFrame({
    'Digit': np.concatenate([digits, digits]),
    'Percentage': np.concatenate([benford, average]),
    'Series': ['Benford'] * 9 + ['Average'] * 9,
})
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=df, x='Digit', y='Percentage', hue='Series', style='Series', marker='o', ax=ax)
ax.set_xlabel('First digit')
ax.set_ylabel('Percentage')
plt.show()


In [ ]:
if 'probabilities' not in globals():
    digits = np.arange(1, 10)
    probabilities = np.log10((digits + 1) / digits)
else:
    digits = np.arange(1, 10)
inv_expected = 300 * probabilities
inv_observed = np.array([86, 48, 23, 32, 24, 36, 19, 18, 14])
df_inv = pd.DataFrame({
    'Digit': np.concatenate([digits, digits]),
    'Frequency': np.concatenate([inv_expected, inv_observed]),
    'Series': ['Expected'] * 9 + ['Observed'] * 9,
})
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=df_inv, x='Digit', y='Frequency', hue='Series', style='Series', marker='o', ax=ax)
ax.set_xlabel('First digit')
ax.set_ylabel('Frequency')
plt.show()


Calculation of the test statistic


In [ ]:
# Expected
np.round(inv_expected, 2)


In [ ]:
# Obs - Exp squared
np.round((inv_observed - inv_expected)**2, 2)


In [ ]:
# Obs - Exp squared divided by expected
np.round(((inv_observed - inv_expected)**2 / inv_expected), 2)
sum(np.round(((inv_observed - inv_expected)**2 / inv_expected), 2))


Critical region


In [ ]:
df = 8
x = np.arange(0, 20.1, 0.1)
tail_start = 15.51
y = chi2.pdf(x, df=df)
fig, ax = plt.subplots(figsize=(8, 4))
display(ax.plot(x, y, color=ADA_PLOT_COLORS['blue']))
mask = x >= tail_start
display(ax.fill_between(x[mask], y[mask], color=ADA_PLOT_COLORS['blue'], alpha=1.0))
ax.set_xlabel('x')
ax.set_ylabel('density')
plt.show()


Using the built-in test


In [ ]:
(lambda _obs, _p: chisquare(f_obs=_obs, f_exp=_obs.sum() * (_p / _p.sum())))(np.asarray(inv_observed, dtype=float), np.asarray(probabilities, dtype=float))


In [ ]:
first_digit = pd.DataFrame({
    'digit': np.arange(1, 10),
    'probability': np.log10(1 + 1 / np.arange(1, 10)),
    'position': 'First',
})
second_digit = pd.DataFrame({
    'digit': np.arange(0, 10),
    'probability': [sum(np.log10(1 + 1 / (10 * np.arange(1, 10) + d))) for d in range(10)],
    'position': 'Second',
})
third_digit = pd.DataFrame({
    'digit': np.arange(0, 10),
    'probability': [sum(np.log10(1 + 1 / (10 * np.arange(10, 100) + d))) for d in range(10)],
    'position': 'Third',
})
benford_data = pd.concat([first_digit, second_digit, third_digit], ignore_index=True)
fig, axes = plt.subplots(3, 1, figsize=(7, 10), sharex=False)
for idx, position in enumerate(['First', 'Second', 'Third']):
    subset = benford_data[benford_data['position'] == position]
    display(axes[idx].plot(subset['digit'], subset['probability'], marker='o'))
    display(axes[idx].set_title(position))
    display(axes[idx].set_xlabel('Digit'))
    display(axes[idx].set_ylabel('Probability'))
plt.tight_layout()
plt.show()


In [ ]:
# The ordinary business-day baseline gives the expected mix of journal-entry
# types. The final two working days provide the observed counts for a focused
# period-end screen.
je_categories = [
    "Sales entries",
    "Purchase entries",
    "Cash receipts/payments",
    "Payroll entries",
    "Inventory and cost-of-sales entries",
    "Accruals and deferrals",
    "Reclassifications",
    "Manual adjustments",
]
je_baseline = np.array([0.26, 0.22, 0.16, 0.07, 0.10, 0.08, 0.06, 0.05])
je_observed = np.array([82, 74, 51, 24, 38, 63, 42, 46])
je_expected = je_observed.sum() * je_baseline
je_summary = pd.DataFrame({
    "category": je_categories,
    "observed": je_observed,
    "baseline_probability": je_baseline,
    "expected": je_expected,
    "chi_square_component": (je_observed - je_expected) ** 2 / je_expected,
})
display(je_summary)
je_chi_square = je_summary["chi_square_component"].sum()
je_df = len(je_categories) - 1
je_critical_value = chi2.ppf(0.95, df=je_df)
je_p_value = chi2.sf(je_chi_square, df=je_df)
pd.Series({
    "chi_square": je_chi_square,
    "degrees_of_freedom": je_df,
    "critical_value": je_critical_value,
    "p_value": je_p_value,
})
chisquare(f_obs=je_observed, f_exp=je_expected)
